# Week 2 — BANKING77 learning-rate search (Kunj)
6 real training runs: 3 methods (full_ft, head_only, lora_r8) × 2 learning rates, seed 42, 5 epochs each.

**Kaggle settings (right panel):** Accelerator = **GPU T4 x2**, Internet = **On**.
**Run it with Save Version → Save & Run All (Commit)** so it keeps running if you close the tab, and so the checkpoints are kept in the version's Output.

In [ ]:
import os
os.environ['CUDA_VISIBLE_DEVICES'] = '0'   # one T4, matching the proposal
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
!git clone https://github.com/devjp99/lora-on-a-budget.git
%cd lora-on-a-budget
!git checkout kunj/week2-lr-selection
!git log --oneline -3

In [ ]:
!pip uninstall -y -q torchao   # Kaggle ships torchao 0.10, which peft rejects; we don't use it
!pip install -q -r requirements.txt
!pip install -q -e .
os.environ['PYTHONPATH'] = 'src'

## 1. Rebuild manifests and run tests

In [ ]:
!python scripts/make_banking77_manifest.py
!git status --short manifests/   # empty output = identical split
!LORA_BUDGET_ONLINE=1 python -m pytest -q

## 2. Learning-rate search: 6 runs (about 30–45 min total)

In [ ]:
import yaml, time
CKPT = '/kaggle/working/checkpoints' if os.path.isdir('/kaggle/working') else 'checkpoints'
for name in ['head_only', 'lora_r8', 'full_ft']:
    cfg_path = f'configs/banking77/{name}.yaml'
    for lr in yaml.safe_load(open(cfg_path))['train']['learning_rates']:
        print(f'\n===== {name}  lr={lr}  ({time.strftime("%H:%M:%S")}) =====')
        !python scripts/train_banking77.py --config {cfg_path} --lr {lr} --seed 42 --phase lr_search --ckpt-root {CKPT}

## 3. Pick the learning rate for each method

In [ ]:
!python scripts/select_lr_banking77.py
import pandas as pd
pd.read_csv('results/train_ledger.csv')[['run_id','status','best_epoch','tune_accuracy','tune_macro_f1','peak_alloc_mb','train_s','train_examples_per_s','checkpoint_mb']]

## 4. Save results for download (checkpoints stay in /kaggle/working/checkpoints, not in git)

In [ ]:
import shutil
out_dir = '/kaggle/working' if os.path.isdir('/kaggle/working') else '.'
shutil.make_archive(os.path.join(out_dir, 'week2_results'), 'zip', '.', 'results')
print('saved', os.path.join(out_dir, 'week2_results.zip'))
!du -sh {CKPT}/* 2>/dev/null